[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/main/examples/citation_check/citation_check.ipynb)

# Check a claim against the source

A citation is either supported, contradicted, or not in the source. An LLM asked "is this true?" tends to answer in prose and to fill gaps. FRIDA-Decisions gets the source as the state and the claim inside one `choice` question, and code sends low-confidence verdicts to a person.

Keep the claim short. Instructions are cut at 96 tokens, including a short suffix the model was trained with, so a long quotation in the instruction is truncated.

Pattern: [Double-checking citations](https://docs.typesafe.ai/cookbooks/citation_check).


In [1]:
%pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


frida-decisions 0.4.0, loaded on cpu, torch.float32


Three outcomes, plus a review branch that is not a model label. Below 0.5 the choice is shown but a person decides. Several correct choices on this set sit around 0.3, so the cutoff is doing real work.


In [3]:
DATA = [
    {"source": "Возврат товара надлежащего качества возможен в течение 14 дней, если сохранены товарный вид и чек.",
     "claim": "Качественный товар можно вернуть в течение 14 дней при наличии чека.",
     "verdict": "supports"},
    {"source": "Возврат товара надлежащего качества возможен в течение 14 дней, если сохранены товарный вид и чек.",
     "claim": "Вернуть качественный товар можно в любой срок и без чека.",
     "verdict": "contradicts"},
    {"source": "Возврат товара надлежащего качества возможен в течение 14 дней, если сохранены товарный вид и чек.",
     "claim": "Доставка по городу бесплатная.",
     "verdict": "insufficient"},
    {"source": "Курс обмена сдаётся до 18:00, после этого заявки переносятся на следующий рабочий день.",
     "claim": "Заявку на обмен после 18:00 рассмотрят на следующий рабочий день.",
     "verdict": "supports"},
    {"source": "Курс обмена сдаётся до 18:00, после этого заявки переносятся на следующий рабочий день.",
     "claim": "Обмен работает круглосуточно, включая выходные.",
     "verdict": "contradicts"},
    {"source": "Гарантия на чайник — один год. Сколы колбы от удара гарантия не покрывает.",
     "claim": "Если колба треснула от удара, гарантия это покрывает.",
     "verdict": "contradicts"},
    {"source": "Гарантия на чайник — один год. Сколы колбы от удара гарантия не покрывает.",
     "claim": "Гарантия на чайник длится один год.",
     "verdict": "supports"},
    {"source": "Гарантия на чайник — один год. Сколы колбы от удара гарантия не покрывает.",
     "claim": "В комплекте есть запасной фильтр.",
     "verdict": "insufficient"},
]
VERDICTS = {
    "supports": "источник подтверждает утверждение",
    "contradicts": "источник противоречит утверждению",
    "insufficient": "в источнике нет данных, чтобы судить",
}
MIN_CONFIDENCE = 0.5


def request_for(source, claim):
    return {
        "state": source,
        "questions": {"verdict": {
            "type": "choice",
            "instructions": "Утверждение: «" + claim + "». Как к нему относится источник?",
            "criteria": VERDICTS,
        }},
    }


def decide(answer):
    if answer["confidence"] < MIN_CONFIDENCE:
        action = "review"
    else:
        action = answer["choice"]
    return {"action": action, "choice": answer["choice"], "confidence": answer["confidence"]}


`review` is the branch taken when confidence is below the cutoff, so an uncertain answer is not stored as a fact. Choice accuracy counts every row. Auto accuracy counts only the rows the code acted on.


In [4]:
def run(judge):
    choice_hits = 0
    auto = 0
    auto_hits = 0
    for row in DATA:
        answer = judge(request_for(row["source"], row["claim"]))["answers"]["verdict"]
        decision = decide(answer)
        if answer["choice"] == row["verdict"]:
            choice_hits += 1
        if decision["action"] == "review":
            mark = "review"
        elif decision["action"] == row["verdict"]:
            mark = "ok"
            auto += 1
            auto_hits += 1
        else:
            mark = "miss"
            auto += 1
        print(f"{mark:6} choice {answer['choice']:14} gold {row['verdict']:14} "
              f"p={answer['confidence']:.2f}  action {decision['action']}")
        print(f"       {row['claim']}")
    print(f"\nchoice agrees with the label in {choice_hits}/{len(DATA)}")
    if auto:
        print(f"auto at confidence >= {MIN_CONFIDENCE}: {auto}/{len(DATA)}, accuracy {auto_hits}/{auto}")
    else:
        print(f"nothing cleared confidence {MIN_CONFIDENCE}")


In [5]:
run(judge)


ok     choice supports       gold supports       p=0.86  action supports
       Качественный товар можно вернуть в течение 14 дней при наличии чека.
review choice contradicts    gold contradicts    p=0.35  action review
       Вернуть качественный товар можно в любой срок и без чека.
review choice contradicts    gold insufficient   p=0.22  action review
       Доставка по городу бесплатная.
ok     choice supports       gold supports       p=0.53  action supports
       Заявку на обмен после 18:00 рассмотрят на следующий рабочий день.
review choice contradicts    gold contradicts    p=0.27  action review
       Обмен работает круглосуточно, включая выходные.
ok     choice contradicts    gold contradicts    p=0.69  action contradicts
       Если колба треснула от удара, гарантия это покрывает.
review choice supports       gold supports       p=0.29  action review
       Гарантия на чайник длится один год.
review choice contradicts    gold insufficient   p=0.33  action review
       В ком